# Tutorial of DiCE with the `dice-ml` package

Taken from [https://interpret.ml/DiCE/readme.html](https://interpret.ml/DiCE/readme.html).

To install the package.
```
pip install dice-ml
```

In [ ]:
!pip install dice-ml

In [2]:
# Sklearn imports
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# DiCE imports
import dice_ml
from dice_ml.utils import helpers  # helper functions

## Loading dataset
We use the "adult income" dataset from UCI Machine Learning Repository ([https://archive.ics.uci.edu/ml/datasets/adult](https://archive.ics.uci.edu/ml/datasets/adult)). 
We transform the data as described in dice_ml.utils.helpers module.

In [3]:
dataset = helpers.load_adult_income_dataset()
dataset.head()

,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,28,Private,Bachelors,Single,White-Collar,White,Female,60,0
1,30,Self-Employed,Assoc,Married,Professional,White,Male,65,1
2,32,Private,Some-college,Married,White-Collar,White,Male,50,0
3,20,Private,Some-college,Single,Service,White,Female,35,0
4,41,Self-Employed,Some-college,Married,White-Collar,White,Male,50,0


In [4]:
d = dice_ml.Data(dataframe=dataset, continuous_features=['age', 'hours_per_week'], outcome_name='income')

## Training a custom ML model
Below, we build an ML model using scikit-learn to demonstrate how our methods can work with any sklearn model. 

In [5]:
target = dataset["income"]

# Split data into train and test
datasetX = dataset.drop("income", axis=1)
x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = ["age", "hours_per_week"]
categorical = x_train.columns.difference(numerical)

# We create the preprocessing pipelines for both numeric and categorical data.
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', categorical_transformer, categorical)]
)

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', RandomForestClassifier())]
               )

# Fit the model to the data
model = clf.fit(x_train, y_train)

In [6]:
# Provide the trained ML model to DiCE's model object
m = dice_ml.Model(model=model, backend='sklearn')

## Generate diverse counterfactuals

In [7]:
# Initiate DiCE
exp_random = dice_ml.Dice(d, m, method="random")

In [8]:
# Select instances
query_instances = x_train[4:6]

# Generate counterfactuals
dice_exp_random = exp_random.generate_counterfactuals(query_instances, total_CFs=4, desired_class="opposite", verbose=False)

100%|███████████████████████████████████████████████| 2/2 [00:00<00:00, 11.79it/s]


In [9]:
dice_exp_random.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,27,Private,School,Single,Blue-Collar,White,Male,40,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,79,-,Assoc,-,-,-,-,-,1
1,80,-,Assoc,-,-,-,-,-,1
2,69,-,Assoc,-,-,-,-,-,1
3,57,-,Assoc,-,-,-,-,-,1


Query instance (original outcome : 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,31,Self-Employed,Some-college,Married,Sales,Other,Male,60,1



Diverse Counterfactual set (new outcome: 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,66,-,-,Separated,-,-,-,-,0
1,-,-,-,-,-,-,-,30,0
2,-,-,School,-,White-Collar,-,-,-,0
3,-,Government,Doctorate,-,-,-,-,-,0


### Selecting the features to vary
Here, you can ensure that DiCE varies only features that it makes sense to vary.

In [10]:
# Generate counterfactuals
dice_exp_random = exp_random.generate_counterfactuals(
        query_instances, total_CFs=4, desired_class="opposite",
        features_to_vary=['workclass', 'education', 'occupation', 'hours_per_week'])

100%|███████████████████████████████████████████████| 2/2 [00:00<00:00, 10.47it/s]


In [11]:
dice_exp_random.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,27,Private,School,Single,Blue-Collar,White,Male,40,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Assoc,-,White-Collar,-,-,96,1
1,-,-,Bachelors,-,White-Collar,-,-,94,1
2,-,-,Bachelors,-,Sales,-,-,79,1
3,-,-,Assoc,-,White-Collar,-,-,97,1


Query instance (original outcome : 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,31,Self-Employed,Some-college,Married,Sales,Other,Male,60,1



Diverse Counterfactual set (new outcome: 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Bachelors,-,-,-,-,93,0
1,-,-,-,-,Blue-Collar,-,-,43,0
2,-,Government,School,-,-,-,-,-,0
3,-,-,-,-,-,-,-,33,0


### Choosing feature ranges
Since the features are sampled randomly, they can freely vary across their range. 
In the below example, we show how range of continuous features can be controlled using *permitted_range* parameter that can now be passed during CF generation.

In [12]:
# generate counterfactuals
dice_exp_random = exp_random.generate_counterfactuals(
    query_instances, total_CFs=4, desired_class="opposite",
    permitted_range={'age': [22, 50], 'hours_per_week': [40, 60]})

100%|███████████████████████████████████████████████| 2/2 [00:00<00:00, 10.82it/s]


In [13]:
dice_exp_random.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,27,Private,School,Single,Blue-Collar,White,Male,40,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,39,-,-,-,Professional,-,-,53,1
1,49,-,-,Widowed,-,-,-,-,1
2,41,-,Prof-school,-,-,-,-,55,1
3,37,-,-,-,-,-,-,59,1


Query instance (original outcome : 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,31,Self-Employed,Some-college,Married,Sales,Other,Male,60,1



Diverse Counterfactual set (new outcome: 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,40,-,-,-,Professional,-,-,-,0
1,-,Government,HS-grad,-,-,-,-,-,0
2,32,-,-,-,-,-,-,-,0
3,23,-,-,-,-,-,-,52,0


## Excercises
- Choose a multiclass dataset and repeat the process. 
- *Try* and repeat everything with PyTorch, with both datasets. Good luck.